# **Pooling**

In Convolutional Neural Networks (CNNs), pooling is the process of shrinking feature maps to make them smaller, reducing the computational load, and helping the model recognize objects regardless of their rotation, scale, or location.

Think of pooling like taking a few steps back from a large painting. You lose the tiny, individual details, but the main subject stands out much more clearly.

## 1. Sliding-Window Pooling Types

These pooling layers slide a small window (usually 2x2 pixels) across the image to summarize it.

### Max Pooling
Max Pooling slides a window across the image and keeps only the single largest (brightest) value in that window, throwing the rest away.

* **Pros:** Extracts sharp features like edges, points, and high-contrast lines. It ignores background noise.
* **Cons:** Throws away a massive amount of data (75% for a 2x2 window) and ignores subtle background textures.

### Average Pooling
Average Pooling slides a window across the image, adds all the pixel values inside, and calculates their average.

* **Pros:** Smooths the downsampling process and preserves background textures and patterns.
* **Cons:** Dilutes sharp features. A bright edge pixel mixed with dark background pixels gets washed out.

### L2 Pooling (Root-Mean-Square Pooling)
L2 Pooling squares all the values in the window, averages them, and takes the square root of the sum:

$$\text{L2 Pooling} = \sqrt{\frac{x_1^2 + x_2^2 + \dots + x_N^2}{N}}$$

* **Pros:** Keeps the sharp contrast of Max Pooling while retaining the smooth background details of Average Pooling.
* **Cons:** Computationally slow and expensive because of the multiple math steps (squares, sums, roots) required for every window.

## 2. Does Pooling Blur Edges? (The Contrast Effect)

The impact of pooling on image edges depends entirely on the type of pooling used.

### Average Pooling Blurs Edges
Because it blends all pixels in the window, a sharp edge pixel is mixed with the surrounding background pixels, diluting its strength.
* **Math Example (2x2 grid with an edge):**
  ```text
  [  0   0 ]  <-- Dark background
  [ 10  10 ]  <-- Bright edge
  ```
  $$\text{Average} = \frac{0 + 0 + 10 + 10}{4} = \mathbf{5} \quad (\text{The edge becomes blurry and faded})$$

### Max Pooling Sharpens Edges
Because it is a "winner-takes-all" method, it keeps only the highest value, maintaining the high contrast.
* **Math Example (Same 2x2 grid):**
  ```text
  [  0   0 ]  <-- Dark background
  [ 10  10 ]  <-- Bright edge
  ```
  $$\text{Maximum} = \max(0, 0, 10, 10) = \mathbf{10} \quad (\text{The edge remains sharp and clear})$$
* **Side Effect:** While Max Pooling doesn't blur edges, it expands them, making the bright features look slightly "fatter" or shifted.

## 3. Global Pooling (Dimension Reduction)

Instead of using a sliding window, Global Pooling collapses the entire height and width of a feature map into a single value.

### Why We Use Global Pooling
Historically, CNNs ended with a `Flatten` layer followed by a giant `Dense` layer. If the final layer outputted 512 channels of $7 \times 7$ images, flattening them resulted in 25,088 features. Connecting this to a Dense layer required over 25 million weights, causing the model to be massive and overfit.

Global Average Pooling (GAP) collapses each $7 \times 7$ map down to 1 average number. This reduces the features from 25,088 down to just 512, saving millions of parameters.

### Global Average Pooling (GAP)
Takes the average of all pixels in the entire feature map. It is the modern standard for closing out CNN architectures (like ResNet).

### Global Max Pooling (GMP)
Takes the absolute peak pixel value from the entire feature map. It is useful when you want to know if a specific feature appears *anywhere* in the image, regardless of how small it is.

* **Global Pooling Pros:** Drastically slashes parameters, prevents overfitting, and allows the model to accept images of varying input sizes.
* **Global Pooling Cons:** Destroys all spatial (location) context. The model will know *what* object is in the image, but not *where* it is located.

## 4. Keras Code Implementations

Here is how to import and define all the discussed pooling methods in Keras:

```python
from tensorflow.keras import layers
import tensorflow as tf

# 1. Standard Max Pooling (2x2 window)
max_pool = layers.MaxPooling2D(pool_size=(2, 2))

# 2. Standard Average Pooling (2x2 window)
avg_pool = layers.AveragePooling2D(pool_size=(2, 2))

# 3. Custom L2 Pooling (Built using AveragePooling and Math functions)
l2_pool = layers.Lambda(lambda x: tf.sqrt(layers.AveragePooling2D(pool_size=(2, 2))(tf.square(x))))

# 4. Global Average Pooling (Used at the end of modern CNNs)
global_avg_pool = layers.GlobalAveragePooling2D()

# 5. Global Max Pooling
global_max_pool = layers.GlobalMaxPooling2D()
```